# Stage 2 — Data Cleaning & Wrangling
### Maven Fuzzy Factory (Online Toy Store) · Tier A pipeline

> **Purpose.** Turn the validated raw tables into analysis-ready `data/processed/` tables. The raw data is unusually clean (Stage 1 found no duplicate PKs and Stage 2's rule checks find no violations), so this stage is deliberately **light and fully documented** — every transformation is explicit and logged.

This notebook drives [`src/cleaning.py`](../src/cleaning.py) and [`src/validation.py`](../src/validation.py); the logic lives in the modules. See [`DOCS/STRUCTURE.md`](../DOCS/STRUCTURE.md) §2 and [`DOCS/IMPLEMENTATION_PLAN.md`](../DOCS/IMPLEMENTATION_PLAN.md) §5.

**What Stage 2 does here:**
1. **Type enforcement** — ids → int64, flags → bool, low-cardinality strings → category.
2. **Null strategy** — the 83,328 untagged sessions become an explicit `traffic_channel` (paid / organic / direct); utm_* nulls → `'direct'`, http_referer nulls → `'(none)'`.
3. **Derived fields** — gross margin on orders & order_items.
4. **Sessionization** — one row per session (entry/exit page, pageview count, duration, bounce, converted).
5. **Business-rule gate** — validation must pass before writing.

**Stage 2 gate (STRUCTURE.md):** null strategy documented per column · duplicates handled · correct dtypes · business rules pass · cleaning log exists · cleaned data saved to `data/processed/`.

## 1. Setup — load raw, then validate BEFORE transforming
We re-ingest the raw tables (Stage 1) and run the business-rule checks on the raw data first. Cleaning only proceeds if the raw data is sane.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import ingestion, cleaning, validation

frames, _ = ingestion.ingest_all()
print("Loaded raw tables:", ", ".join(frames))

Loaded raw tables: website_sessions, website_pageviews, orders, order_items, order_item_refunds, products


## 2. Business-rule validation (the gate)
13 rules covering non-negativity, cost ≤ price, referential integrity, refund ≤ price paid, temporal sanity, and item-count consistency. All must pass before we clean.

In [2]:
rules = validation.check_business_rules(frames)
assert validation.all_passed(rules), "Business-rule validation failed — stop and inspect."
validation.rules_dataframe(rules)

,rule,passed,n_violations,detail
0,orders.price_usd >= 0,True,0,
1,orders.cogs_usd >= 0,True,0,
2,order_items.price_usd >= 0,True,0,
3,order_item_refunds.refund_amount_usd >= 0,True,0,
4,orders: cogs_usd <= price_usd,True,0,
5,order_items: cogs_usd <= price_usd,True,0,
6,pageviews.website_session_id in sessions,True,0,
7,orders.website_session_id in sessions,True,0,
8,order_items.order_id in orders,True,0,
9,refunds.order_item_id in order_items,True,0,


## 3. The null story — why we don't just drop or blanket-fill
All nulls live in `website_sessions`. The *pattern* of nulls is itself information: sessions with a UTM tag are paid; sessions with no UTM but a referrer are organic/referral; sessions with neither are direct type-ins. We encode that as `traffic_channel` instead of discarding it.

In [3]:
s = frames["website_sessions"]
print("Nulls per column (raw):")
print(s.isnull().sum()[lambda x: x > 0].to_string())
print("\nNull cross-tab (utm vs referer) -> the three channels:")
print(pd.crosstab(s["utm_source"].isnull(), s["http_referer"].isnull(),
                  rownames=["utm_null"], colnames=["referer_null"]))

Nulls per column (raw):
utm_source      83328
utm_campaign    83328
utm_content     83328
http_referer    39917

Null cross-tab (utm vs referer) -> the three channels:
referer_null   False  True 
utm_null                   
False         389543      0
True           43411  39917


## 4. Clean all tables + sessionize
`clean_all()` applies the per-table cleaners, derives margin, and builds `session_summary` from the pageview clickstream. It returns the processed frames and a structured changelog.

In [4]:
processed, log = cleaning.clean_all(frames)
cleaning._print_summary(log, rules)

STAGE 2 - CLEANING SUMMARY

[website_sessions]  (472,871 rows)
   - Derived traffic_channel {paid=389543, organic=43411, direct=39917} from utm/referer null pattern
   - Filled 83328 null utm_source with 'direct' (untagged traffic)
   - Filled 83328 null utm_campaign with 'direct' (untagged traffic)
   - Filled 83328 null utm_content with 'direct' (untagged traffic)
   - Filled 39917 null http_referer with '(none)'
   - Cast ids->int64, is_repeat_session->bool, utm/device/channel->category

[website_pageviews]  (1,188,124 rows)
   - Cast ids->int64, pageview_url->category

[orders]  (32,313 rows)
   - Cast ids/items_purchased->int64
   - Derived margin_usd = price_usd - cogs_usd and margin_pct

[order_items]  (40,025 rows)
   - Cast ids->int64, is_primary_item->bool
   - Derived margin_usd = price_usd - cogs_usd

[order_item_refunds]  (1,731 rows)
   - Cast ids->int64

[products]  (4 rows)
   - Cast product_id->int64, product_name->string

[session_summary]  (472,871 rows)
   - Session

## 5. Verify the results
### 5a. Traffic channel is now explicit and MECE

In [5]:
sc = processed["website_sessions"]
chan = sc["traffic_channel"].value_counts()
print(chan.to_string())
print(f"\nTotal = {chan.sum():,} (matches session count: {len(sc):,})")
assert chan.sum() == len(sc)
print("No remaining nulls in sessions:", int(sc.isnull().sum().sum()) == 0)

traffic_channel
paid       389543
organic     43411
direct      39917

Total = 472,871 (matches session count: 472,871)
No remaining nulls in sessions: True


### 5b. Dtypes are enforced (compact + correct)

In [6]:
for name in ["orders", "order_items", "session_summary"]:
    print(f"\n[{name}]")
    print(processed[name].dtypes.to_string())


[orders]
order_id                       int64
created_at            datetime64[us]
website_session_id             int64
user_id                        int64
primary_product_id             int64
items_purchased                int64
price_usd                    float64
cogs_usd                     float64
margin_usd                   float64
margin_pct                   float64

[order_items]
order_item_id               int64
created_at         datetime64[us]
order_id                    int64
product_id                  int64
is_primary_item              bool
price_usd                 float64
cogs_usd                  float64
margin_usd                float64

[session_summary]
website_session_id             int64
session_start         datetime64[us]
session_end           datetime64[us]
n_pageviews                    int64
entry_url                   category
exit_url                    category
duration_seconds               int64
is_bounce                       bool
is_converted      

### 5c. Sessionization sanity — converted sessions must equal the order count
A session is `is_converted` if it appears in `orders`. Because each order maps to exactly one session, converted sessions should equal the number of orders (32,313) — an independent cross-check on the sessionization.

In [7]:
ss = processed["session_summary"]
n_conv = int(ss["is_converted"].sum())
n_orders = len(processed["orders"])
print(f"Converted sessions : {n_conv:,}")
print(f"Orders             : {n_orders:,}")
assert n_conv == n_orders, "Mismatch — investigate sessionization/join."
print(f"Bounce rate        : {ss['is_bounce'].mean():.1%}")
print(f"Overall conversion : {ss['is_converted'].mean():.2%}")
print("\nTop entry (landing) pages:")
print(ss["entry_url"].value_counts().head(8).to_string())

Converted sessions : 32,313
Orders             : 32,313
Bounce rate        : 44.8%
Overall conversion : 6.83%

Top entry (landing) pages:
entry_url
/home         137576
/lander-2     131170
/lander-3      79000
/lander-5      68166
/lander-1      47574
/lander-4       9385
/billing           0
/billing-2         0


### 5d. Margin fields look sane

In [8]:
o = processed["orders"]
print(o[["price_usd", "cogs_usd", "margin_usd", "margin_pct"]].describe().round(2).to_string())
assert (o["margin_usd"] >= 0).all(), "Negative margin found!"
print("\nAll order margins non-negative:", bool((o["margin_usd"] >= 0).all()))

       price_usd  cogs_usd  margin_usd  margin_pct
count   32313.00  32313.00    32313.00    32313.00
mean       59.99     22.36       37.64        0.63
std        17.81      6.24       11.75        0.02
min        29.99      9.49       20.50        0.61
25%        49.99     19.49       30.50        0.61
50%        49.99     19.49       30.50        0.61
75%        59.99     22.49       37.50        0.64
max       109.98     41.98       69.00        0.68

All order margins non-negative: True


## 6. Persist the processed layer + changelog
Write all seven tables to `data/processed/` as Parquet (columnar, compressed, dtype-preserving) and the changelog to `logs/cleaning_log.json`.

In [9]:
paths = cleaning.write_processed(processed)
clog = cleaning.write_changelog(log)
print(f"Wrote {len(paths)} processed tables:")
for p in paths:
    print("   -", p.relative_to(PROJECT_ROOT))
print("Changelog:", clog.relative_to(PROJECT_ROOT))

Wrote 7 processed tables:
   - data\processed\website_sessions.parquet
   - data\processed\website_pageviews.parquet
   - data\processed\orders.parquet
   - data\processed\order_items.parquet
   - data\processed\order_item_refunds.parquet
   - data\processed\products.parquet
   - data\processed\session_summary.parquet
Changelog: logs\cleaning_log.json


## 7. Stage 2 gate — status

| Gate item | Status |
|---|---|
| Missing-value strategy documented per column | ✅ utm_* → `'direct'`, referer → `'(none)'`, plus derived `traffic_channel` (paid/organic/direct) |
| Duplicates assessed & handled | ✅ none exist (0 dup PKs / 0 exact dup rows — confirmed Stage 1) |
| All columns correct dtypes | ✅ ids→int64, flags→bool, low-card→category, datetimes parsed |
| Business-rule validations pass | ✅ 13/13 rules pass |
| Cleaning log exists (what/how many/why) | ✅ `logs/cleaning_log.json` |
| Cleaned data saved to `data/processed/` | ✅ 7 Parquet tables (incl. derived `session_summary`) |

**Next stage:** [Stage 3 — EDA](../DOCS/IMPLEMENTATION_PLAN.md) — hypothesis-driven, one exhibit per issue-tree branch (Acquisition → Conversion → Monetization → Leakage), each with an action title and a "So What", styled per [`DOCS/DESIGN.md`](../DOCS/DESIGN.md).